# Занятие 10, пара 2. Многослойная сеть в PyTorch


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## От одного нейрона к сети

В первой паре каждый пиксель имел свой вес. Теперь добавим скрытый слой: сеть сама соберёт
промежуточные признаки. Разберём вычисления, обучение и кривые, не перескакивая сразу к готовому fit.
Работаем с десятью цифрами UCI 32×32 из 09.2, на CPU. OpenCV, GPU и скачивание данных не нужны.

Если PyTorch есть, обучим небольшую MLP. Если нет, выполним те же вычисления прямого прохода в NumPy
с **предоставленными обученными весами**, исследуем ошибки и влияние скрытого слоя. Это собственный
запуск предсказаний, но не собственное обучение. Все «Ваш ход» доступны в обоих маршрутах.
Результат — понятная схема сети, журнал/анализ обучения и проверка выбора модели по validation.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


## 1. Скрытый слой на трёх числах

Первый Linear превращает входные признаки в новые взвешенные суммы. Для одной картинки:
$h_{raw}=xW_1+b_1$. Затем **ReLU** оставляет положительное значение и заменяет отрицательное нулём:
$h=\max(0,h_{raw})$. Второй слой: $z=hW_2+b_2$.

Зачем промежуточная нелинейность? Без неё $(xW_1+b_1)W_2+b_2$ сворачивается в один линейный слой.
Дополнительное матричное умножение само по себе не даёт сети нелинейную границу.
Ниже веса маленькие и заданы вручную **для объяснения**, они ещё не обучены.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json

small_x = np.array([1., 2.])
small_w1 = np.array([[1., -1., .5], [0., 1., -1.]])
small_b1 = np.array([0., 0., .5])
hidden_raw = small_x @ small_w1 + small_b1
hidden = np.maximum(0, hidden_raw)
small_w2 = np.array([[1.,0.], [0.,1.], [1.,-1.]])
small_b2 = np.array([.1,-.1])
small_logits = hidden @ small_w2 + small_b2
print('До ReLU:', hidden_raw, 'После:', hidden, 'Логиты:', small_logits)


ReLU не обещает «оставить всё полезное»: она пропускает положительные значения текущей суммы.
Обучение изменяет веса перед ней, а значит — какие признаки и для каких изображений включатся.
Вычисление от картинки к логитам называется **прямым проходом**. Обратный проход вычислит,
как ошибка зависит от каждого параметра через цепочку этих операций.


## 2. Десять выходов — десять логитов

Для взаимоисключающих цифр хотим десять оценок. Победитель — индекс максимального логита.
Softmax переводит их в доли с суммой 1: $p_k=e^{z_k}/\sum_j e^{z_j}$.
Для устойчивости сначала вычтем максимальный логит: общий множитель сократится.
Это не десять независимых sigmoid: повышение одной вероятности уменьшает остальные.

Потеря для истинного класса y — $-\log p_y$. В PyTorch `CrossEntropyLoss` объединяет устойчивый
log-softmax и выбор истинного класса. На вход ей передаём **логиты**, а не уже посчитанные softmax-вероятности.
Цели — целые номера классов 0…9.


In [ ]:
def softmax_rows(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)

example_logits = np.array([[2.,1.,0.]])
example_probabilities = softmax_rows(example_logits)
print('Вероятности:', example_probabilities, 'Сумма:', example_probabilities.sum())
print('Loss при истинном классе 0:', -np.log(example_probabilities[0,0]))


### ✏️ Ваш ход — Логиты испугали коллегу

Коллега получил логиты [1002,1001,1000] и решил, что сеть точно
увереннее, чем при [2,1,0]. Вычислите `my_probabilities` для обоих рядов устойчивым softmax.
Запишите `my_losses` для истинного класса 2. Что изменилось, а что нет? Почему прямой exp от 1002 опасен?
Не используйте случайное обрезание логитов: оно может изменить отношения между классами.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.2.1', my_probabilities, my_losses)


**Мой вывод:** …


## 3. Выбираем доступный маршрут

Tensor похож на массив NumPy, но может хранить историю вычислений и вычислять градиенты.
`dtype=torch.float32` — вещественные числа для входов и весов; `torch.long` — целые метки.
`device='cpu'` означает процессор. Видеокарту в этом опыте не используем.

Следующая служебная ячейка только выбирает маршрут. Установка PyTorch — отдельная подготовка среды,
а не задание на пару. `FORCE_NUMPY=True` позволит посмотреть запасной маршрут даже при установленном пакете.


In [ ]:
FORCE_NUMPY = False
TORCH_AVAILABLE = False
if not FORCE_NUMPY:
    try:
        import torch
        from torch import nn
        TORCH_AVAILABLE = True
    except (ImportError, OSError) as error:
        print('PyTorch недоступен:', type(error).__name__)
print('Маршрут:', 'собственное обучение на CPU' if TORCH_AVAILABLE else 'NumPy: анализ предоставленных весов')


## 4. Что делает autograd — на одном параметре

Пусть $L=(2w-1)^2$. При w=0.5 loss=0 и производная=0 — не очень интересно.
Возьмём w=0: по формуле производная $2(2w-1)\cdot2=-4$.
`requires_grad=True` просит запоминать операции для производной. `loss.backward()` проходит граф
назад и складывает производную в `w.grad`. **Вес пока не обновился.** Это отдельно делает оптимизатор.
Повторный backward на новом графе добавит градиент к уже имеющемуся: поэтому нужен zero_grad.
`detach()` отделяет число от графа, `.numpy()` даёт массив для вывода; на CPU этого достаточно.


In [ ]:
manual_gradient = 2 * (2 * 0. - 1) * 2
if TORCH_AVAILABLE:
    torch_w = torch.tensor(0., requires_grad=True)
    torch_loss = (2 * torch_w - 1) ** 2
    torch_loss.backward()
    print('Формула:', manual_gradient, 'Autograd:', torch_w.grad.item())
    print('Вес после backward всё ещё:', torch_w.item())
else:
    eps = 1e-5
    numerical_gradient = (((2*eps-1)**2) - ((-2*eps-1)**2)) / (2*eps)
    print('Формула:', manual_gradient, 'Численно:', numerical_gradient)
    print('Autograd в этом маршруте не запускался.')


## 5. Одинаковая задача и одинаковые части данных

Используем train=1934 и validation=946 из UCI. Test=1797 сохраняем для итогового сравнения с CNN:
в этой тетради по нему не выбираем архитектуру, эпоху или скорость обучения.
Изображения уже подготовлены авторами: бинарные, нормализованные по размеру и положению.
Это не обещание распознать произвольную фотографию тетради.

Flatten превращает [N,32,32] в [N,1024]. Затем Linear(1024,32), ReLU, Linear(32,10).
Параметров: 1024×32+32 + 32×10+10 = 33130. У каждого скрытого нейрона свой вес на каждый пиксель.
В CNN следующего блока будут общие ядра для разных позиций; здесь их ещё нет.


In [ ]:
digits = np.load(DATA_DIR / 'digits' / 'optdigits32.npz', allow_pickle=False)
train_images = digits['train_images'].astype(np.float32)
val_images = digits['validation_images'].astype(np.float32)
y_train = digits['train_labels'].astype(np.int64)
y_val = digits['validation_labels'].astype(np.int64)
x_train = train_images.reshape(len(train_images), -1)
x_val = val_images.reshape(len(val_images), -1)
print('Вход:', x_train.shape, 'Метки:', y_train.shape)
print('Число параметров:', 1024*32+32 + 32*10+10)


`nn.Sequential` проводит вход через перечисленные слои по порядку: для простой цепочки этого
достаточно, свой класс пока не нужен. `nn.Linear(in,out)` хранит матрицу размера [out,in] и смещение:
поэтому при переносе в NumPy матрицу транспонируем.
`torch.manual_seed` фиксирует генератор начальных весов/перестановок в данном окружении;
на разных версиях и платформах побитовое совпадение не обещаем.
`torch.from_numpy` оборачивает массив в CPU-тензор (память общая, исходник не меняем).
`set_num_threads(2)` ограничивает вычисления двумя потоками, чтобы небольшой опыт не занимал все ядра.

Сначала создаём модель и делаем только прямой проход двух картинок. Случайные начальные логиты
не обязаны угадывать цифры. Выход [2,10] — десять оценок для каждой из двух картинок.


In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(42)
    torch.set_num_threads(2)
    model = nn.Sequential(nn.Flatten(), nn.Linear(1024,32), nn.ReLU(), nn.Linear(32,10)).to('cpu')
    train_x_tensor = torch.from_numpy(train_images)
    train_y_tensor = torch.from_numpy(y_train)
    val_x_tensor = torch.from_numpy(val_images)
    val_y_tensor = torch.from_numpy(y_val)
    initial_logits = model(train_x_tensor[:2])
    print('Форма выхода:', tuple(initial_logits.shape))
else:
    print('Модель для NumPy загрузим после разбора цикла; структура та же: 1024 → 32 → 10.')


## 6. Один шаг оптимизатора, затем повторение

Создадим loss и **SGD** — обычный градиентный спуск без momentum. Ему передаются обучаемые параметры
`model.parameters()` и скорость 0.1. Мини-пачка — 128 изображений. На каждой пачке:

1. `optimizer.zero_grad()` очищает накопленные градиенты.
2. `model(batch_x)` вычисляет логиты; `loss_fn(logits,batch_y)` — среднюю потерю.
3. `loss.backward()` считает градиенты всех весов через цепочку слоёв.
4. `optimizer.step()` обновляет параметры против градиента.

Ни одна из этих строк не заменяет остальные. Сначала выполним один пробный шаг отдельно. Для нескольких пачек `torch.randperm(N)` перемешивает
индексы train. Последняя пачка может быть короче — это нормально.
Ниже сначала создаём объекты, затем пройдём один шаг по частям. Перед полным опытом вернём модель
в исходное состояние: пробный шаг не станет скрытой дополнительной эпохой.


In [ ]:
if TORCH_AVAILABLE:
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=.1)
    batch_size = 128
    epochs = 12
    print('Пачка:', batch_size, 'Эпохи:', epochs)


### Один mini-batch: сначала вход и loss

Берём первые 128 изображений только для разбора. Все строки ниже уже входят в описанный шаг:
вход имеет форму [128,32,32], выход — [128,10]. Пока веса не меняются.
В NumPy-маршруте этот фрагмент читаем вместе с преподавателем; собственная работа с весами будет ниже.


In [ ]:
if TORCH_AVAILABLE:
    demo_x = train_x_tensor[:128]
    demo_y = train_y_tensor[:128]
    optimizer.zero_grad()
    demo_logits = model(demo_x)
    demo_loss = loss_fn(demo_logits, demo_y)
    print('Форма:', tuple(demo_logits.shape), 'Loss до шага:', demo_loss.item())


Теперь `backward`: смотрим градиент **одного** веса первого Linear.
В Sequential слой 0 — Flatten, 1 — первый Linear, 2 — ReLU, 3 — второй Linear.
Выражение `.weight.grad[0,100]` — производная по весу первого скрытого нейрона для пикселя 100.
`.item()` извлекает одно число для печати. Наличие grad не означает изменение weight.


In [ ]:
if TORCH_AVAILABLE:
    demo_weight_before = model[1].weight[0,100].item()
    demo_loss.backward()
    demo_gradient = model[1].weight.grad[0,100].item()
    print('Градиент:', demo_gradient)
    print('Вес пока:', model[1].weight[0,100].item(), 'Был:', demo_weight_before)


И только теперь `step` меняет параметры. Сравните изменение выбранного веса с -lr·grad.
Повторно считаем loss на **той же пачке**, чтобы увидеть эффект одного шага. Это ещё не качество
на новых изображениях: для этого есть validation. Изменение одного конкретного веса может оказаться нулём.


In [ ]:
if TORCH_AVAILABLE:
    optimizer.step()
    demo_weight_after = model[1].weight[0,100].item()
    with torch.no_grad():
        demo_new_logits = model(demo_x)
        demo_new_loss = loss_fn(demo_new_logits, demo_y).item()
    print('Изменение веса:', demo_weight_after - demo_weight_before, 'Ожидали:', -.1 * demo_gradient)
    print('Loss после шага:', demo_new_loss)
    # Пробу закончили. Полный опыт начинается с тех же начальных весов и seed.
    torch.manual_seed(42)
    model = nn.Sequential(nn.Flatten(), nn.Linear(1024,32), nn.ReLU(), nn.Linear(32,10)).to('cpu')
    optimizer = torch.optim.SGD(model.parameters(), lr=.1)


После эпохи оцениваем train и validation **одними текущими весами**.
`model.train()` включает режим обучения; `model.eval()` — режим оценки (особенно важно для будущих
Dropout/BatchNorm). `torch.no_grad()` отключает построение графа там, где обучение не требуется.
Эти две команды решают разные задачи. У нашей простой MLP eval сам по себе ответы не меняет.

Сохраняем историю и веса с наименьшей validation-loss. `state_dict()` содержит параметры;
`value.detach().clone()` делает отдельную копию, иначе дальнейшее обучение изменило бы сохранённые значения.
`load_state_dict(best_state)` в конце вернёт выбранные веса. Для точности сравниваем argmax с метками:
в PyTorch `dim=1` соответствует NumPy `axis=1`, `.float()` переводит True/False в 1/0.
Пока не смотрим test. В NumPy-маршруте эта ячейка не обучает: ниже будет подписанный журнал готового запуска.


In [ ]:
if TORCH_AVAILABLE:
    history = []
    best_loss = float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(train_images))
        for start in range(0, len(order), batch_size):
            indices = order[start:start + batch_size]
            batch_x = train_x_tensor[indices]
            batch_y = train_y_tensor[indices]
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_logits = model(train_x_tensor)
            val_logits = model(val_x_tensor)
            train_loss = loss_fn(train_logits, train_y_tensor).item()
            val_loss = loss_fn(val_logits, val_y_tensor).item()
            train_pred = train_logits.argmax(dim=1)
            val_pred = val_logits.argmax(dim=1)
            train_correct = (train_pred == train_y_tensor).float()
            val_correct = (val_pred == val_y_tensor).float()
            train_acc = train_correct.mean().item()
            val_acc = val_correct.mean().item()
        history.append([epoch, train_loss, val_loss, train_acc, val_acc])
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            best_state = {}
            for name, value in model.state_dict().items():
                best_state[name] = value.detach().clone()
        print(epoch, 'train loss:', round(train_loss,3), 'val loss:', round(val_loss,3))
    model.load_state_dict(best_state)
    w1 = model[1].weight.detach().numpy().T.copy()
    b1 = model[1].bias.detach().numpy().copy()
    w2 = model[3].weight.detach().numpy().T.copy()
    b2 = model[3].bias.detach().numpy().copy()
    history = np.array(history)
    run_origin = 'own_cpu_training'
else:
    saved = np.load(DATA_DIR / 'digits' / 'mlp_reference.npz', allow_pickle=False)
    w1 = saved['w1']; b1 = saved['b1']; w2 = saved['w2']; b2 = saved['b2']
    history = saved['history']
    best_epoch = int(saved['best_epoch'])
    run_origin = 'provided_training_run'
    print('Загружены предоставленные веса и журнал. Собственное обучение не выполнялось.')
print('Происхождение:', run_origin, 'Выбранная эпоха:', best_epoch)


## 7. Прямой проход в NumPy — одинаковый для обоих маршрутов

Повторяем знакомые два слоя с полученными весами. Так видно, что обученная сеть — конкретная
последовательность вычислений, а не магический объект библиотеки. `argmax(axis=1)` выбирает класс
для каждой строки логитов. Сравниваем с validation, не с test.

Кривые показывают изменение потери и доли верных ответов. Если train продолжает улучшаться, а
validation устойчиво ухудшается, есть сигнал переобучения. Одна неровная точка ещё не доказывает его.
Если разрыва нет, не надо «находить переобучение» ради нужного ответа.
SGD обновляет веса по разным мини-пачкам, поэтому loss между эпохами не обязана убывать монотонно.
В предоставленном запуске есть скачки, но лучшая validation-loss достигнута на эпохе 12.
Это описание конкретного журнала, а не обещание такого же числа при любых версиях/настройках.


In [ ]:
hidden_raw = x_val @ w1 + b1
hidden = np.maximum(0, hidden_raw)
val_logits_numpy = hidden @ w2 + b2
val_predictions = val_logits_numpy.argmax(axis=1)
val_accuracy = (val_predictions == y_val).mean()
majority = np.bincount(y_train).argmax()
baseline_accuracy = (y_val == majority).mean()
print('Validation accuracy:', val_accuracy, 'Ориентир:', baseline_accuracy)
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history[:,0], history[:,1], label='train')
axes[0].plot(history[:,0], history[:,2], label='validation')
axes[0].axvline(best_epoch, color='gray', linestyle='--')
axes[0].set(xlabel='Эпоха', ylabel='Cross entropy', title='Потери')
axes[1].plot(history[:,0], history[:,3], label='train')
axes[1].plot(history[:,0], history[:,4], label='validation')
axes[1].set(xlabel='Эпоха', ylabel='Accuracy', title=run_origin)
for ax in axes: ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


### ✏️ Ваш ход — Какую модель отдадим дальше?

В отдельном учебном журнале validation-loss по эпохам 1…5:
[.8,.5,.42,.47,.51], train-loss: [.7,.4,.25,.15,.1].
Напишите `choose_epoch(values)`: номер первой эпохи с наименьшим validation-loss, нумерация с 1;
для пустого списка — None. Укажите, какие **веса** надо сохранить и почему последняя train-loss не решает вопрос.
Функция выбирает по одному заранее заданному критерию, не ищет test-метрику.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.2.2', choose_epoch)


**Мой вывод:** …


## 8. Посмотрим на ошибки крупно

Берём первые ошибки в фиксированном порядке, чтобы не выбирать только смешные или удобные.
Рядом вероятность победителя: высокая уверенность может сопровождать неверный ответ.
Если ошибок мало, покажем сколько есть; отсутствие ошибок на этом срезе не доказывает безошибочность системы.


In [ ]:
wrong_indices = np.flatnonzero(val_predictions != y_val)
probabilities = softmax_rows(val_logits_numpy)
fig, axes = plt.subplots(2,4,figsize=(12,6))
for ax in axes.ravel(): ax.axis('off')
for ax, index in zip(axes.ravel(), wrong_indices[:8]):
    predicted = val_predictions[index]
    ax.imshow(val_images[index], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'Верно {y_val[index]}, ответ {predicted}\np={probabilities[index,predicted]:.2f}')
plt.tight_layout(); plt.show()
print('Всего ошибок validation:', len(wrong_indices))


### ✏️ Ваш ход — Что изменит выключение одного нейрона?

В маленьком скрытом слое h=[2,0,1], матрица второго слоя
W=[[1,-1],[2,2],[-1,1]], смещение [0,0]. Посчитайте исходные логиты.
Затем **по очереди** выключите каждый нейрон (замените только его h на 0) и получите
`my_disabled_logits` размера [3,2]. Для каждого случая объясните изменение ответа; равенство логитов — ничья,
а не «уверенная победа нулевого класса». Какой нейрон в этом примере не влияет и почему?
Не изменяйте основной массив hidden: дальнейший общий код должен остаться прежним.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.2.3', my_original_logits, my_disabled_logits)


**Мой вывод:** …


## 9. Сохраним результат с честной подписью

В NPZ сохраняем массивы без сериализации Python-объектов. В JSON — параметры опыта и происхождение.
Предоставленные веса можно исследовать, но нельзя назвать собственным обучением для ЛР.
Никаких скачанных исполняемых моделей здесь нет; `allow_pickle=False` при чтении остаётся хорошей привычкой.


In [ ]:
output_dir = WORKSPACE_DIR / 'outputs' / 'mlp_digits'
output_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output_dir / 'weights.npz', w1=w1, b1=b1, w2=w2, b2=b2)
record = {
    'origin': run_origin, 'architecture': [1024,32,10], 'best_epoch': int(best_epoch),
    'selection': 'minimum validation cross entropy; first tie',
    'validation_accuracy': float(val_accuracy), 'test_evaluated_here': False,
    'data': 'UCI Optdigits original tra/cv; same split as 09.2',
}
(output_dir / 'result.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
np.savetxt(output_dir / 'history.csv', history, delimiter=',',
           header='epoch,train_loss,validation_loss,train_accuracy,validation_accuracy', comments='')
print(output_dir)


## Дальше — CNN и ЛР6

Полное задание — в `docs/ЛР6.md`, 20 баллов по прежним критериям. Сегодня начинаем подготовку:
понимаем обучение и проверяем среду. **MLP не заменяет CNN**, которая требуется в итоговой ЛР6.
Свёрточную сеть и работу со своими данными разберём в блоках 11–12. Итоговое сравнение с ЛР5 —
одна задача, те же группы и test. Подготовленные цифры — учебный пример, не автоматическая смена темы проекта.
Если для самостоятельного обучения нет среды, заранее обсудите с преподавателем доступный компьютер
или запуск вашего кода с сохранением журнала; не подменяйте эксперимент чужими весами.

На этой паре test намеренно не открываем. Классика уже дала один ориентир в 09.2; будущую CNN
выбираем по validation, затем сравним замороженные решения на том же test. Повторный подбор по его ошибкам
превратит test в ещё одну validation-часть.

Резерв: выберите один скрытый нейрон **до просмотра результата**, обнулите его активации в копии hidden,
пересчитайте validation-ответы и покажите 4 изменившихся примера. Влияние на этих данных не равно
причинному объяснению всех решений модели.


## Полезные первоисточники

- [PyTorch: автоматическое дифференцирование](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).
- [PyTorch: цикл оптимизации](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).
- [CrossEntropyLoss: логиты и целевые классы](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).
- [Dive into Deep Learning: многослойные перцептроны](https://d2l.ai/chapter_multilayer-perceptrons/index.html).

Происхождение цифр — `course_support/data/digits/SOURCE.md`; параметры предоставленного запуска,
версии среды и контрольная сумма исходных данных — `course_support/data/digits/MLP_RUN.json`.
Для запуска без PyTorch этот файл и NPZ уже включены в комплект.
